## 係数とホーナー法による f, f' の計算

In [1]:
# f(x) の係数 a[i] は x^i の係数 (a0, a1, ..., a11)
coeffs = [30, -30, -15, 5, 10, -1, 1, 3, -1, -2, -1, 1]

def horner(a, x):
    # ホーナー法で f(x)=p と f'(x)=dp を同時計算 ( **, pow は使わない )
    k = len(a) - 1
    p = a[k]      # f
    dp = 0.0      # f'
    i = k - 1
    while i >= 0:
        dp = dp * x + p
        p = p * x + a[i]
        i -= 1
    return p, dp

def deflate(a, r):
    # 減次: a(x) を (x - r) で割った商の係数を返す (組立除法)
    k = len(a) - 1
    q = [0] * k          # 商は k-1 次
    q[k - 1] = a[k]
    i = k - 2
    while i >= 0:
        q[i] = a[i + 1] + r * q[i + 1]
        i -= 1
    return q


## ニュートン法 + 減次で全解を求める

In [2]:
def newton(a, x0, tol=1e-13, maxiter=200):
    # ニュートン法。収束すれば根を、しなければ None を返す
    x = complex(x0)
    for _ in range(maxiter):
        p, dp = horner(a, x)
        if dp == 0:
            return None
        dx = p / dp
        x = x - dx
        if abs(dx) < tol:          # 収束判定: |x_{n+1}-x_n| < tol
            return x
    return None

def find_all_roots(a):
    orig = [complex(c) for c in a]
    cur = orig[:]
    # 複素初期値を含む種々の開始点（実根・複素根の双方に到達させる）
    starts = [0.5+0.5j, -0.5+0.5j, 1+1j, -1-1j, 1j, 2+0j, -2+0j,
              0.1+0.9j, 3-1j, -3+1j, 0.5-1.5j, -1.5-0.5j]
    roots = []
    while len(cur) - 1 > 1:
        r = None
        for s in starts:
            r = newton(cur, s)
            if r is not None:
                break
        if r is None:
            break
        r = newton(orig, r) or r        # 元の多項式で仕上げ研磨
        roots.append(r)
        cur = deflate(cur, r)
    if len(cur) - 1 == 1:               # 最後の1次式
        roots.append(-cur[0] / cur[1])
    return roots

all_roots = find_all_roots(coeffs)


## 実数解

In [3]:
real_roots = sorted(r.real for r in all_roots if abs(r.imag) < 1e-8)
print(f"実数解 {len(real_roots)} 個:")
for x in real_roots:
    val, _ = horner(coeffs, x)
    print(f"  x = {x:+.12f}   f(x) = {val.real:+.2e}")


実数解 5 個:
  x = -1.414213562373   f(x) = -3.20e-14
  x = +1.000000000000   f(x) = +0.00e+00
  x = +1.379729661461   f(x) = +7.11e-15
  x = +1.414213562373   f(x) = +0.00e+00
  x = +1.442249570307   f(x) = +7.11e-15


## 発展: 複素解を含む全解

In [4]:
print(f"全 {len(all_roots)} 解:")
for r in sorted(all_roots, key=lambda z: (z.real, z.imag)):
    val, _ = horner(coeffs, r)
    print(f"  x = {r.real:+.10f} {r.imag:+.10f}i   |f(x)| = {abs(val):.2e}")


全 11 解:
  x = -1.4142135624 -0.0000000000i   |f(x)| = 3.20e-14
  x = -1.1162247438 +0.8109847472i   |f(x)| = 3.43e-14
  x = -1.1162247438 -0.8109847472i   |f(x)| = 1.78e-13
  x = -0.7211247852 -1.2490247665i   |f(x)| = 3.20e-14
  x = -0.7211247852 +1.2490247665i   |f(x)| = 3.20e-14
  x = +0.4263599130 -1.3122008853i   |f(x)| = 1.12e-14
  x = +0.4263599130 +1.3122008853i   |f(x)| = 1.12e-14
  x = +1.0000000000 -0.0000000000i   |f(x)| = 5.32e-44
  x = +1.3797296615 -0.0000000000i   |f(x)| = 7.11e-15
  x = +1.4142135624 +0.0000000000i   |f(x)| = 1.63e-30
  x = +1.4422495703 +0.0000000000i   |f(x)| = 7.11e-15
